<a href="https://colab.research.google.com/github/manipoint/llm-finetuning/blob/main/prefreance_align_traning_dpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from transformers import AutoModelForCausalLM, TrainingArguments,Trainer,AutoTokenizer
from peft import LoraConfig,TaskType,get_peft_model
from datasets import load_dataset
import torch

In [5]:
import zipfile,os,json

adapter_zip = "/content/drive/MyDrive/llmfine-tuning/homebudget_lora_adapter.zip"
adapter_dir = "/content/homebudget_lora_adapter"

os.makedirs(adapter_dir,exist_ok=True)

with zipfile.ZipFile(adapter_zip,"r") as zip_ref:
  zip_ref.extractall(adapter_dir)

print(os.listdir(adapter_dir))

['chat_template.jinja', 'tokenizer_config.json', 'checkpoint-45', 'checkpoint-25', 'README.md', 'tokenizer.json', 'adapter_config.json', 'adapter_model.safetensors']


In [6]:
config_path = os.path.join(adapter_dir,"adapter_config.json")
with(open(config_path,"r")) as f:
  adapter_config = json.load(f)



In [7]:
print(json.dumps(adapter_config, indent=2))

{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 32,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "lora_ga_config": null,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "peft_type": "LORA",
  "peft_version": "0.19.1",
  "qalora_group_size": 16,
  "r": 16,
  "rank_pattern": {},
  "revision": null,
  "target_modules": [
    "k_proj",
    "down_proj",
    "o_proj",
    "gate_proj",
    "q_proj",
    "v_proj",
    "up_proj"
  ],
  "target_parameters": null,
  "task_type": "CAUSAL_LM",
  "trainabl

In [8]:
from torch._higher_order_ops import auto_functionalize
from peft import PeftModel
print("GPU available:",torch.cuda.is_available())


base_model_name = adapter_config["base_model_name_or_path"]
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"





GPU available: True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [9]:
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=dtype,
    device_map="auto",
)

base_model.config.use_cache = False
base_model.config.pad_token_id = tokenizer.pad_token_id

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [10]:
!pip install torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 104.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [11]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    adapter_dir,
    is_trainable=True
)

model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [12]:
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.3 MB/s eta 0:00:00


In [13]:
try:
  import trl
  print("TRL version:", trl.__version__)
except ImportError:
    print("TRL installed nahi hai")

TRL version: 1.6.0


In [14]:
train_path = "/content/drive/MyDrive/homebudget_sft_dataset/homebudget_messages_train.jsonl"
validation_path = "/content/drive/MyDrive/homebudget_sft_dataset/homebudget_messages_validation.jsonl"

sft_train = load_dataset(
    "json",
    data_files=train_path,
    split="train"
)

sft_validation = load_dataset(
    "json",
    data_files=validation_path,
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [15]:
print("Train rows:", len(sft_train))
print("Validation rows:", len(sft_validation))
print("Columns:", sft_train.column_names)
print(json.dumps(sft_train[0], indent=2, ensure_ascii=False))

Train rows: 116
Validation rows: 20
Columns: ['id', 'topic', 'messages', 'source', 'format_version']
{
  "id": "homebudget-0022",
  "topic": "budget",
  "messages": [
    {
      "role": "system",
      "content": "You are HomeBudget Assistant. Answer questions about the HomeBudget app, its product rules, ledger design, architecture, APIs, validation, and implementation plan. Be accurate, practical, and grounded in the HomeBudget product requirements. Do not invent features that are outside the current phase; clearly say when something belongs to a later phase."
    },
    {
      "role": "user",
      "content": "Can a user have multiple active budgets for the same period?"
    },
    {
      "role": "assistant",
      "content": "The current recommended rule says a user may have one active budget per currency and overlapping period. This decision can be revisited later, but it should be enforced unless the product requirements change."
    }
  ],
  "source": "HomeBudget Product Requi

In [16]:
for index in range(min(3,len(sft_train))):
  print(f"\n--- Record {index} ---")
  print(json.dumps(
        sft_train[index],
        indent=2,
        ensure_ascii=False
    ))


--- Record 0 ---
{
  "id": "homebudget-0022",
  "topic": "budget",
  "messages": [
    {
      "role": "system",
      "content": "You are HomeBudget Assistant. Answer questions about the HomeBudget app, its product rules, ledger design, architecture, APIs, validation, and implementation plan. Be accurate, practical, and grounded in the HomeBudget product requirements. Do not invent features that are outside the current phase; clearly say when something belongs to a later phase."
    },
    {
      "role": "user",
      "content": "Can a user have multiple active budgets for the same period?"
    },
    {
      "role": "assistant",
      "content": "The current recommended rule says a user may have one active budget per currency and overlapping period. This decision can be revisited later, but it should be enforced unless the product requirements change."
    }
  ],
  "source": "HomeBudget Product Requirements and Architecture",
  "format_version": "messages-v1"
}

--- Record 1 ---
{


In [17]:
def convert_sft_to_preference(example):
  message = example["messages"]

  if message[-1]["role"] != "assistant":
    raise ValueError(f'Last message assistant nahi hai: {example["id"]}')
  return {
      "prompt": message[:-1],
      "chosen": message[-1]
  }




In [18]:
dpo_train = sft_train.map(
    convert_sft_to_preference,
    remove_columns=["messages", "source", "format_version"]
)
dpo_validation = sft_validation.map(
    convert_sft_to_preference,
    remove_columns=["messages", "source", "format_version"]
)

Map:   0%|          | 0/116 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [19]:
print(dpo_train)
print(json.dumps(
    dpo_train[0],
    indent=2,
    ensure_ascii=False
))

Dataset({
    features: ['id', 'topic', 'prompt', 'chosen'],
    num_rows: 116
})
{
  "id": "homebudget-0022",
  "topic": "budget",
  "prompt": [
    {
      "content": "You are HomeBudget Assistant. Answer questions about the HomeBudget app, its product rules, ledger design, architecture, APIs, validation, and implementation plan. Be accurate, practical, and grounded in the HomeBudget product requirements. Do not invent features that are outside the current phase; clearly say when something belongs to a later phase.",
      "role": "system"
    },
    {
      "content": "Can a user have multiple active budgets for the same period?",
      "role": "user"
    }
  ],
  "chosen": {
    "content": "The current recommended rule says a user may have one active budget per currency and overlapping period. This decision can be revisited later, but it should be enforced unless the product requirements change.",
    "role": "assistant"
  }
}


In [20]:
def prepare_dpo_example(example):
  messages = example["messages"]
  prompt_text = tokenizer.apply_chat_template(
      messages[:-1],
      tokenize=False,
      add_generation_prompt=True
  )
  chosen_text = messages[-1]["content"]

  return{
      "id":example["id"],
      "topic":example["topic"],
      "prompt":prompt_text,
      "chosen":chosen_text,
  }

In [21]:
dpo_train = sft_train.map(
    prepare_dpo_example,
    remove_columns=sft_train.column_names
)

dpo_validation = sft_validation.map(
    prepare_dpo_example,
    remove_columns=sft_validation.column_names
)

print(dpo_train)
print(dpo_train[0])


Map:   0%|          | 0/116 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'topic', 'prompt', 'chosen'],
    num_rows: 116
})
{'id': 'homebudget-0022', 'topic': 'budget', 'prompt': '<|system|>\nYou are HomeBudget Assistant. Answer questions about the HomeBudget app, its product rules, ledger design, architecture, APIs, validation, and implementation plan. Be accurate, practical, and grounded in the HomeBudget product requirements. Do not invent features that are outside the current phase; clearly say when something belongs to a later phase.</s>\n<|user|>\nCan a user have multiple active budgets for the same period?</s>\n<|assistant|>\n', 'chosen': 'The current recommended rule says a user may have one active budget per currency and overlapping period. This decision can be revisited later, but it should be enforced unless the product requirements change.'}


In [22]:
@torch.no_grad()
def generate_rejected(example):
  inputs = tokenizer(
      example["prompt"],
      return_tensors="pt",
  ).to(model.device)
  prompt_lenght = inputs['input_ids'].shape[1]
  with model.disable_adapter():
    genrated = model.generate(
        **inputs,
        max_new_tokens=128,
        min_new_tokens=10,
        do_sample=True,
        temperature=1.2,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

  rejected_text = tokenizer.decode(
     genrated[0][prompt_lenght:],
     skip_special_tokens=True,

 ).strip()

  return {"rejected": rejected_text}

In [23]:
dpo_test = dpo_train.select(range(3)).map(
    generate_rejected,
    load_from_cache_file=False,
    desc="Generating test rejected answers"
)

Generating test rejected answers:   0%|          | 0/3 [00:00<?, ? examples/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [24]:
for row in dpo_test:
    print("\nQUESTION:")
    print(row["prompt"].split("<|user|>")[-1].split("</s>")[0].strip())

    print("\nCHOSEN:")
    print(row["chosen"])

    print("\nREJECTED:")
    print(row["rejected"])

    print("\n" + "=" * 80)


QUESTION:
Can a user have multiple active budgets for the same period?

CHOSEN:
The current recommended rule says a user may have one active budget per currency and overlapping period. This decision can be revisited later, but it should be enforced unless the product requirements change.

REJECTED:
As per your query, yes, a user can have multiple active budgets for the same period within HomeBudget app. It depends on your business requirement, whether you want the user to manage multiple projects or activities under the same budget at once. In our previous version, all of your active budgets belong to the same financial calendar, but that feature will be changed in the upcoming release.

In such a scenario where you have multiple projects and activities that need the same budgeting mechanism, then you could create two separate HomeBudget apps, one for each project or activity. Each project or activity would get its


QUESTION:
What is the recommended implementation order after wallets

In [25]:
from transformers import set_seed
set_seed(42)
model.eval()
model.config.use_cache = True


In [26]:
dpo_train = dpo_train.map(generate_rejected,load_from_cache_file=False,desc = "Generating train rejected answers")
dpo_validation = dpo_validation.map(generate_rejected,load_from_cache_file=False,desc = "Generating validation rejected answers")

Generating train rejected answers:   0%|          | 0/116 [00:00<?, ? examples/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Generating validation rejected answers:   0%|          | 0/20 [00:00<?, ? examples/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

In [27]:
empty_train = [
    row["id"] for row in dpo_train if not row["rejected"].strip()
]

identical_train = [
    row["id"] for row in dpo_train if row["chosen"].strip() == row["rejected"].strip()
]
print("Train rows:", len(dpo_train))
print("Validation rows:", len(dpo_validation))
print("Empty rejected:", empty_train)
print("Identical answers:", identical_train)

Train rows: 116
Validation rows: 20
Empty rejected: []
Identical answers: []


In [28]:
!pip install --upgrade datasets

In [33]:
import os
import json

dpo_output_dir = "/content/drive/MyDrive/homebudget_dpo_dataset"
os.makedirs(dpo_output_dir, exist_ok=True)

dpo_train_path = os.path.join(dpo_output_dir, "homebudget_dpo_train.jsonl")
dpo_validation_path = os.path.join(dpo_output_dir, "homebudget_dpo_validation.jsonl")


def save_jsonl(dataset, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in dataset:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


save_jsonl(dpo_train, dpo_train_path)
save_jsonl(dpo_validation, dpo_validation_path)

print("Saved train:", dpo_train_path)
print("Saved validation:", dpo_validation_path)

Saved train: /content/drive/MyDrive/homebudget_dpo_dataset/homebudget_dpo_train.jsonl
Saved validation: /content/drive/MyDrive/homebudget_dpo_dataset/homebudget_dpo_validation.jsonl


In [34]:
model.config.use_cache = False
model.train()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [36]:
import transformers
import datasets
import peft
import trl
import inspect

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("PyTorch:", torch.__version__)

print("\nDPOConfig signature:")
print(inspect.signature(trl.DPOConfig.__init__))

Transformers: 5.10.2
Datasets: 4.0.0
PEFT: 0.19.1
TRL: 1.6.0
PyTorch: 2.11.0+cu128

DPOConfig signature:
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 1e-06, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, g

In [37]:
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

GPU available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [38]:
import numpy as np

def get_pair_length(row):
    chosen_tokens = tokenizer(
        row["prompt"] + row["chosen"],
        add_special_tokens=False
    )["input_ids"]

    rejected_tokens = tokenizer(
        row["prompt"] + row["rejected"],
        add_special_tokens=False
    )["input_ids"]

    return max(len(chosen_tokens), len(rejected_tokens))


train_lengths = [get_pair_length(row) for row in dpo_train]
validation_lengths = [get_pair_length(row) for row in dpo_validation]

all_lengths = train_lengths + validation_lengths

print("Minimum:", min(all_lengths))
print("Median:", int(np.percentile(all_lengths, 50)))
print("95th percentile:", int(np.percentile(all_lengths, 95)))
print("Maximum:", max(all_lengths))

Minimum: 168
Median: 234
95th percentile: 242
Maximum: 263


In [39]:
dpo_train_ready = dpo_train.remove_columns(["id", "topic"])

dpo_validation_ready = dpo_validation.remove_columns(["id", "topic"])

print(dpo_train_ready)
print(dpo_train_ready.column_names)

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 116
})
['prompt', 'chosen', 'rejected']


In [40]:
model.config.use_cache = False
model.enable_input_require_grads()
model.train()

model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [47]:
from trl import DPOConfig,DPOTrainer

dpo_output_dir = "/content/drive/MyDrive/llmfine-tuning/homebudget_dpo_adapter"

dpo_config = DPOConfig(
    output_dir=dpo_output_dir,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    beta = 0.1,
    max_length=320,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },
    warmup_steps=0.1,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    save_total_limit=2,
    report_to="none",
    seed=42
)


In [50]:
from trl import DPOTrainer
traniner = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_train_ready,
    eval_dataset=dpo_validation_ready,
    processing_class=tokenizer

)

Adding EOS to train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

In [52]:
torch.cuda.empty_cache()
baseline_metrics = traniner.evaluate()
print(json.dumps(baseline_metrics,indent=2))

Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
No log,0.693147,0,1.831412,0.000000,-3.370693,-3.394239,0.610244,0.000000,0.000000,0.000000,0.000000,-83.233591,-306.727721


{
  "eval_loss": 0.6931473016738892,
  "eval_entropy": 1.8314121425151826,
  "eval_num_tokens": 0.0,
  "eval_logits/chosen": -3.3706933146975677,
  "eval_logits/rejected": -3.3942388658389184,
  "eval_mean_token_accuracy": 0.6102438479661941,
  "eval_rewards/chosen": 0.0,
  "eval_rewards/rejected": 0.0,
  "eval_rewards/accuracies": 0.0,
  "eval_rewards/margins": 0.0,
  "eval_logps/chosen": -83.23359069824218,
  "eval_logps/rejected": -306.7277206420898
}


In [53]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
tranin_result = traniner.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,0.218333,0.168368,1.823270,44964.000000,-3.329792,-3.360817,0.607339,-0.038887,-1.804120,1.000000,1.765232,-83.622464,-324.768919
2,0.088998,0.099922,1.820451,89928.000000,-3.314345,-3.348169,0.602869,-0.067380,-2.450206,1.000000,2.382826,-83.907390,-331.229783


In [54]:
print(json.dumps(tranin_result.metrics,indent=2))
print("Peek memory:",round(torch.cuda.max_memory_allocated()/1024**3,2),"GB")

{
  "train_runtime": 128.2559,
  "train_samples_per_second": 1.809,
  "train_steps_per_second": 0.234,
  "total_flos": 685340518219776.0,
  "train_loss": 0.2684791112939517,
  "epoch": 2.0
}
Peek memory: 2.59 GB


In [55]:
final_metrics = traniner.evaluate()

comparison_keys = [
    "eval_loss",
    "eval_rewards/accuracies",
    "eval_rewards/margins",
    "eval_rewards/chosen",
    "eval_rewards/rejected"
]

for key in comparison_keys:
  print(
        key,
        "| before:", baseline_metrics.get(key),
        "| after:", final_metrics.get(key)
    )


Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.088998,0.099922,2,1.820451,89928.000000,-3.314345,-3.348169,0.602869,-0.067380,-2.450206,1.000000,2.382826,-83.907390,-331.229783


eval_loss | before: 0.6931473016738892 | after: 0.09992227703332901
eval_rewards/accuracies | before: 0.0 | after: 1.0
eval_rewards/margins | before: 0.0 | after: 2.3828262984752655
eval_rewards/chosen | before: 0.0 | after: -0.06737996174488217
eval_rewards/rejected | before: 0.0 | after: -2.450206255912781


In [56]:
final_adapter_dir = (
    "/content/drive/MyDrive/llmfine-tuning/"
    "homebudget_dpo_final_adapter"
)
traniner.save_model(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print(os.listdir(final_adapter_dir))

['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'ref', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin']


In [57]:
import shutil

zip_path = shutil.make_archive(final_adapter_dir,"zip",final_adapter_dir)
print("Saved:",zip_path)

Saved: /content/drive/MyDrive/llmfine-tuning/homebudget_dpo_final_adapter.zip


In [58]:
import gc
del traniner
del model
del base_model

gc.collect()
torch.cuda.empty_cache()

In [61]:
tokenizer = AutoTokenizer.from_pretrained(final_adapter_dir)
base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto",
)

dpo_model = PeftModel.from_pretrained(
    base_model,final_adapter_dir
)
dpo_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [66]:
def ask_homebudget(question):
    messages = [
        {
            "role": "system",
            "content": (
                "You are HomeBudget Assistant. Answer accurately and "
                "only according to the current HomeBudget product rules."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(dpo_model.device)

    with torch.no_grad():
        output = dpo_model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    response_tokens = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        response_tokens,
        skip_special_tokens=True
    ).strip()

In [67]:
questions = [
    "Can I directly edit a wallet balance?",
    "How should HomeBudget correct an incorrect transaction?",
    "Can a user have multiple active budgets for an overlapping period?",
    "What should be implemented after wallets?"
]

for question in questions:
    print("\nQuestion:", question)
    print("Answer:", ask_homebudget(question))
    print("-" * 80)

[transformers] Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question: Can I directly edit a wallet balance?


[transformers] Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: No. Wallet balances should not be edited directly. The wallet balance must be calculated from finalized transactions, and changes must be recorded as adjustments or reversals.
--------------------------------------------------------------------------------

Question: How should HomeBudget correct an incorrect transaction?


[transformers] Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: A correction must be reversed, not just adjusted. For example, if a transaction was recorded as -500 but it should have been +500, the correction is +500.
--------------------------------------------------------------------------------

Question: Can a user have multiple active budgets for an overlapping period?


[transformers] Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: No. A user can only have one active budget per financial period.
--------------------------------------------------------------------------------

Question: What should be implemented after wallets?
Answer: After wallets, the user should create a category for household expenses, such as Grocery, Bills, Clothing, and Miscellaneous. The category should have sub-categories like Grocery, Bills, Clothing, and Miscellaneous.
--------------------------------------------------------------------------------
